RAG RETRIEVAL AND RECOMMENDATION PIPELINE

In [3]:
# ============================================================
# AIRLINE RAG - REQUIRED LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
!pip install sentence_transformers
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

np.random.seed(42)

print("All airline RAG libraries loaded successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.3/780.3 kB 23.5 MB/s  0:00:00


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


All airline RAG libraries loaded successfully.


In [11]:
import numpy as np
np.random.seed(42)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd

file_path = '/content/drive/MyDrive/Skytrax_Airline_Reviews_Cleaned.csv'

df = pd.read_csv(file_path)

df.head()

,review_date,over_all_rating,review_header,trip_verified,review_text,aircraft,type_of_traveller,seat_type,route,date_flown,...,ground_service,wifi_connectivity,value_for_money,recommended,airline_name,trip_verified_raw,review_text_clean,review_header_clean,review_text_char_count,review_text_word_count
0,2025-08-04,1.0,showed zero empathy for passengers,verified,My afternoon flight was delayed and eventually...,NaN,business,economy_class,dubai_to_kochi,2025-08-01,...,1.0,1.0,1.0,no,spicejet,Trip Verified,My afternoon flight was delayed and eventually...,showed zero empathy for passengers,788.0,133.0
1,2025-07-21,2.0,Extremely dissatisfied,verified,The flight was postponed 3 times with insuffic...,NaN,family_leisure,economy_class,pune_to_dubai,2025-07-01,...,3.0,NaN,2.0,no,spicejet,Trip Verified,The flight was postponed 3 times with insuffic...,Extremely dissatisfied,611.0,89.0
2,2025-04-19,2.0,Very bad experience,verified,They will charge you for your hand baggage if ...,NaN,business,economy_class,mumbai_to_kolkata,2025-04-01,...,1.0,NaN,2.0,no,spicejet,Trip Verified,They will charge you for your hand baggage if ...,Very bad experience,175.0,31.0
3,2025-03-12,1.0,“What a difficult airline”,verified,The flight was delayed from 15:20 to 16:25. I ...,NaN,business,economy_class,"pune,_india_to_new_delhi,_india",2025-03-01,...,5.0,NaN,1.0,no,spicejet,Trip Verified,The flight was delayed from 15:20 to 16:25. I ...,“What a difficult airline”,523.0,97.0
4,2025-03-02,2.0,“Poor chaotic boarding”,verified,No crowd control and boarding was like cattle ...,boeing_747,solo_leisure,economy_class,mumbai_to_new_delhi,2025-03-01,...,1.0,NaN,2.0,no,spicejet,Trip Verified,No crowd control and boarding was like cattle ...,“Poor chaotic boarding”,466.0,80.0


In [6]:
# Load the skytrax_airline dataset
#df= pd.read_csv("Skytrax_Airline_Reviews_Cleaned.csv")

# Check the available columns
print("Columns:")
print(df.columns.tolist())

print("\nDataset shape:", df.shape)

Columns:
['review_date', 'over_all_rating', 'review_header', 'trip_verified', 'review_text', 'aircraft', 'type_of_traveller', 'seat_type', 'route', 'date_flown', 'seat_comfort', 'cabin_staff_service', 'food_beverages', 'inflight_entertainment', 'ground_service', 'wifi_connectivity', 'value_for_money', 'recommended', 'airline_name', 'trip_verified_raw', 'review_text_clean', 'review_header_clean', 'review_text_char_count', 'review_text_word_count']

Dataset shape: (64676, 24)


# CREATE RAG DOCUMENT

In [7]:
# ============================================================
# PREPARE COLUMNS FOR RAG DOCUMENT CREATION
# ============================================================

# Text columns
text_columns = [
    "airline_name",
    "review_header_clean",
    "review_text_clean",
    "type_of_traveller",
    "seat_type",
    "route",
    "aircraft",
    "trip_verified",
    "date_flown"
]

for col in text_columns:
    df[col] = df[col].fillna("").astype(str)


# Rating / service columns
rating_columns = [
    "over_all_rating",
    "seat_comfort",
    "cabin_staff_service",
    "food_beverages",
    "inflight_entertainment",
    "ground_service",
    "wifi_connectivity",
    "value_for_money"
]

for col in rating_columns:
    df[col] = df[col].fillna("Not rated").astype(str)


# Recommended column
df["recommended"] = (
    df["recommended"]
    .fillna("Unknown")
    .astype(str)
)


# ============================================================
# CREATE DOCUMENT FOR EACH AIRLINE REVIEW
# ============================================================

df["rag_document"] = (
    "Airline: " + df["airline_name"] +
    ". Traveller type: " + df["type_of_traveller"] +
    ". Seat type: " + df["seat_type"] +
    ". Route: " + df["route"] +
    ". Aircraft: " + df["aircraft"] +
    ". Trip verified: " + df["trip_verified"] +
    ". Date flown: " + df["date_flown"] +
    ". Review title: " + df["review_header_clean"] +
    ". Passenger review: " + df["review_text_clean"] +
    ". Overall rating: " + df["over_all_rating"] +
    ". Seat comfort: " + df["seat_comfort"] +
    ". Cabin staff service: " + df["cabin_staff_service"] +
    ". Food and beverages: " + df["food_beverages"] +
    ". Inflight entertainment: " + df["inflight_entertainment"] +
    ". Ground service: " + df["ground_service"] +
    ". WiFi connectivity: " + df["wifi_connectivity"] +
    ". Value for money: " + df["value_for_money"] +
    ". Recommended: " + df["recommended"]
)


# ============================================================
# REMOVE DUPLICATE DOCUMENTS
# ============================================================

rag_documents = (
    df["rag_document"]
    .drop_duplicates()
    .reset_index(drop=True)
)


print("Dataset shape:", df.shape)
print("Number of unique retrieval documents:", len(rag_documents))


# ============================================================
# DISPLAY SAMPLE DOCUMENTS
# ============================================================

for i in range(min(3, len(rag_documents))):
    print(f"\nDocument {i + 1}:")
    print(rag_documents.iloc[i])

Dataset shape: (64676, 25)
Number of unique retrieval documents: 64674

Document 1:
Airline: spicejet. Traveller type: business. Seat type: economy_class. Route: dubai_to_kochi. Aircraft: . Trip verified: verified. Date flown: 2025-08-01. Review title: showed zero empathy for passengers. Passenger review: My afternoon flight was delayed and eventually flew only at 7 PM, completely ruining my day. Even worse, my return flight, which was scheduled for 11 PM, was silently pushed to 5:30 AM — with no notice or communication whatsoever. I was left waiting for hours without any proper updates or support from the airline. It was exhausting and completely disrupted my schedule. I lost sleep for two nights and had to attend an important meeting tired and sleep-deprived, just trying to hold it together. SpiceJet showed zero empathy for passengers. They seem to treat delays like it’s just another day at work, without caring how it affects real people with real commitments. Would not recommend. Ch

GENERATE EMBEDDINGS

In [8]:
# ============================================================
# CREATE DOCUMENT EMBEDDINGS
# ============================================================

from sentence_transformers import SentenceTransformer

# Load the pre-trained embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings using larger batches
document_embeddings = embedding_model.encode(
    rag_documents.tolist(),
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", document_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/506 [00:00<?, ?it/s]

Embedding shape: (64674, 384)


CREATE RAG RETRIEVAL KNOWLEDGE BASE

In [12]:
# ============================================================
# CREATE RAG RETRIEVAL KNOWLEDGE BASE
# ============================================================

# Ensure documents have a clean sequential index
rag_documents = rag_documents.reset_index(drop=True)

# Check that every document has a corresponding embedding
assert len(rag_documents) == len(document_embeddings), \
    "Number of documents does not match number of embeddings."

# Create retrieval dataframe
rag_knowledge_base = pd.DataFrame({
    "document": rag_documents,
    "embedding": [np.asarray(emb) for emb in document_embeddings]
})

# Display knowledge base information
print("RAG knowledge base created successfully.")
print(f"Number of documents: {len(rag_knowledge_base):,}")
print(f"Embedding dimensions: {len(rag_knowledge_base['embedding'].iloc[0])}")

# Preview
display(rag_knowledge_base.head())

RAG knowledge base created successfully.
Number of documents: 64,674
Embedding dimensions: 384


,document,embedding
0,Airline: spicejet. Traveller type: business. S...,"[0.0027511744, -0.049273517, 0.03614247, 0.094..."
1,Airline: spicejet. Traveller type: family_leis...,"[-0.024541633, 0.013457232, 0.033625036, 0.083..."
2,Airline: spicejet. Traveller type: business. S...,"[-0.024411326, 0.0066642654, -0.011241611, 0.0..."
3,Airline: spicejet. Traveller type: business. S...,"[-0.0007082915, -0.058335222, 0.0037701097, 0...."
4,Airline: spicejet. Traveller type: solo_leisur...,"[0.020279003, -0.014852169, 0.01770696, 0.0731..."


Load recommendation LLM

In [13]:
# ============================================================
# LOAD LANGUAGE MODEL FOR RAG GENERATION
# ============================================================

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Lightweight model suitable for local use
model_name = "google/flan-t5-small"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load generation model
generator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Airline recommendation language model loaded successfully.")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Airline recommendation language model loaded successfully.


In [14]:
from sentence_transformers import SentenceTransformer

# Embedding model used for retrieval
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [18]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("Retrieval libraries loaded successfully.")

Retrieval libraries loaded successfully.


RETRIEVE REVIEWS

In [19]:
def retrieve_reviews(query, model, knowledge_base, top_k=5):
    """
    Retrieve the most semantically similar airline reviews
    from the RAG knowledge base.
    """

    # Generate embedding for the user's review
    query_embedding = model.encode(
        query,
        convert_to_numpy=True
    )

    # Stack stored document embeddings
    document_embeddings = np.vstack(
        knowledge_base["embedding"].values
    )

    # Calculate similarity
    similarities = cosine_similarity(
        query_embedding.reshape(1, -1),
        document_embeddings
    )[0]

    # Identify top matching documents
    top_indices = similarities.argsort()[::-1][:top_k]

    # Return matching documents and similarity scores
    results = knowledge_base.iloc[top_indices].copy()

    results["similarity_score"] = similarities[top_indices]

    return results[
        ["document", "similarity_score"]
    ].reset_index(drop=True)

In [20]:
def generate_customer_support_response(
    review_text,
    embedding_model,
    knowledge_base,
    generator_model,
    tokenizer,
    conversation_history="No previous conversation history provided.",
    top_k=5
):
    """
    Retrieve similar airline reviews and generate:

    1. A short summary of the customer's issue.
    2. A summary of relevant conversation history.
    3. A professional and empathetic customer response.
    4. A human-attention assessment.
    5. An internal recommended action.

    Human attention should be considered for issues such as:
    - unresolved refunds;
    - repeated service failures;
    - urgent or ongoing travel disruptions;
    - missed connections;
    - cancellations without rebooking;
    - lost baggage;
    - safety, medical, accessibility, or legal concerns.
    """

    # --------------------------------------------------------
    # STEP 1: RETRIEVE SIMILAR AIRLINE REVIEWS
    # --------------------------------------------------------

    retrieved_reviews = retrieve_reviews(
        query=review_text,
        model=embedding_model,
        knowledge_base=knowledge_base,
        top_k=top_k
    )

    # --------------------------------------------------------
    # STEP 2: PREPARE RETRIEVED CONTEXT
    # --------------------------------------------------------

    context_documents = []

    for index, row in retrieved_reviews.iterrows():
        context_documents.append(
            f"Retrieved case {index + 1}:\n"
            f"{row['document']}\n"
            f"Similarity score: {row['similarity_score']:.4f}"
        )

    retrieved_context = "\n\n".join(context_documents)

    # --------------------------------------------------------
    # STEP 3: CREATE AIRLINE CUSTOMER-SUPPORT PROMPT
    # --------------------------------------------------------

    prompt = f"""
You are an airline customer support assistant.

Your role is to analyse a passenger complaint and prepare a helpful
customer-service response grounded in relevant historical airline reviews.

CURRENT CUSTOMER REVIEW:
{review_text}

CUSTOMER CONVERSATION HISTORY:
{conversation_history}

RETRIEVED HISTORICAL CASES:
{retrieved_context}

Complete the following tasks:

1. Summarise the customer's main issue in no more than two sentences.

2. Summarise any relevant conversation history.
If no previous history was provided, state:
"No previous conversation history was provided."

3. Identify the main airline service area affected.
Examples include:
- Flight disruption
- Refunds
- Ground service
- Baggage handling
- Cabin service
- Booking and ticketing
- Customer communication
- Accessibility assistance
- Safety or medical support

4. Determine whether the complaint requires human attention.

Set Human Attention Required to "Yes" if the complaint involves:
- an unresolved or delayed refund;
- a repeated service failure;
- an urgent or ongoing travel disruption;
- a cancellation without confirmed rebooking;
- a missed or at-risk connection;
- lost or significantly delayed baggage;
- a safety, security, medical, legal, or accessibility concern;
- financial loss requiring investigation;
- a complaint that cannot be resolved using the available information.

Otherwise, set it to "No".

5. Write a professional and empathetic suggested response to the customer.
The response must:
- acknowledge the customer's experience;
- show empathy without admitting unsupported legal liability;
- address the specific issue raised;
- state a practical next step;
- avoid inventing refund status, booking details, flight details,
  compensation eligibility, investigation results, or contact history;
- not promise a refund, compensation, or resolution unless supported
  by the provided information;
- avoid copying personal or airline details from unrelated retrieved cases.

6. Provide a short internal recommended action for the airline team.

Return the answer exactly in the following format:

Customer Issue Summary:
[Short summary]

Conversation History Summary:
[Short history summary]

Service Area:
[Main service area]

Human Attention Required:
[Yes or No]

Escalation Reason:
[Reason for escalation, or "No immediate escalation trigger identified."]

Suggested Customer Response:
[Professional and empathetic response]

Internal Recommended Action:
[Practical internal action]
"""

    # --------------------------------------------------------
    # STEP 4: TOKENIZE THE PROMPT
    # --------------------------------------------------------

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Move inputs to the same device as the generation model
    model_device = next(generator_model.parameters()).device

    inputs = {
        key: value.to(model_device)
        for key, value in inputs.items()
    }

    # --------------------------------------------------------
    # STEP 5: GENERATE THE RESPONSE
    # --------------------------------------------------------

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=300,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2
    )

    # --------------------------------------------------------
    # STEP 6: DECODE THE GENERATED RESPONSE
    # --------------------------------------------------------

    generated_response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return generated_response, retrieved_reviews

In [21]:
new_review = """
The flight was delayed for several hours and the ground staff
did not communicate the reason for the delay. I have contacted
the airline twice, but I have not received a clear response.
"""

conversation_history = """
The customer previously contacted the airline twice regarding
the delay but has not received a clear explanation or resolution.
"""

customer_response, retrieved_reviews = generate_customer_support_response(
    review_text=new_review,
    conversation_history=conversation_history,
    embedding_model=embedding_model,
    knowledge_base=rag_knowledge_base,
    generator_model=generator_model,
    tokenizer=tokenizer,
    top_k=5
)

print("CUSTOMER SUPPORT OUTPUT")
print("=" * 70)
print(customer_response)

print("\nRETRIEVED HISTORICAL REVIEWS")
print("=" * 70)
display(retrieved_reviews)

CUSTOMER SUPPORT OUTPUT
The flight was delayed for several hours and the ground staff did not communicate the reason for the delay

RETRIEVED HISTORICAL REVIEWS


,document,similarity_score
0,Airline: american-airlines. Traveller type: so...,0.728338
1,Airline: united-airlines. Traveller type: busi...,0.712894
2,Airline: united-airlines. Traveller type: fami...,0.689875
3,Airline: ita-airways. Traveller type: couple_l...,0.688609
4,Airline: united-airlines. Traveller type: solo...,0.682492
